In [1]:
import pandas as pd
import numpy as np

import os
import tqdm

import matplotlib.pyplot as plt
import matplotlib.style as style

style.use('tableau-colorblind10')

In [2]:
import torch
from torch.utils.data import DataLoader

from sklearn.preprocessing import FunctionTransformer

import helper as hl
import models as ml
import dataset as ds

In [3]:
from sklearn.metrics import r2_score, explained_variance_score
from sktime.performance_metrics.forecasting import MeanAbsoluteScaledError, MeanAbsolutePercentageError, MeanSquaredError, MeanAbsoluteError

# Define Global Variables

In [4]:
SR_FREQ = '12H'
device = torch.device('cpu')

In [5]:
cli_args = dict(
    # LSTM PARAMS
    data='paloalto',  # dundee | porto | boulder | paloalto
    # rnn_cell='lstm',
    rnn_cell='gru',
    # bi=False, 
    bi=True, 
    hidden_size=24, 
    num_layers=1, 
    fc_layers='24',
    # DATA PARAMS
    sr_freq=SR_FREQ, 
    min_pts=100,
    # TRAINING PARAMS
    bs=32, 
    length=48, 
    stride=1, 
    n_epochs=50, 
    patience=10
)

windowing_params = dict(
    length_min=cli_args['min_pts'], 
    length_max=cli_args['length'], 
    stride=cli_args['stride'],
    rnn_feats=[
        'day_sin', 'day_cos',
        'hour_sin', 'hour_cos', 
        'week_sin', 'week_cos', 
        #
        'power_curr_logdelta',
        'power_next_step1_extrap',
        # 
        'power_curr_std', 
        'power_curr_ema',
        #
        f'downtime_scaled',
        #
        'no_of_sessions_scaled',
        'charging_time',
        'power_curr', 
    ],
    # Extra features to include in the Fully Connected (FC) layer (excl. ```building``` and ```model```)
    fc_feats=[
        'power_output_kW', 
    ],
    y_feats=['power_next'], 
)

# Load Dataset

In [6]:
df_evse_demand_v3 = pd.read_pickle(
    os.path.join(
        '../data', 'pkl', 
        f"{cli_args['data']}_data.demand_{cli_args['sr_freq']}_{cli_args['min_pts']}_points.enriched.v4.pickle"
    )
).sort_index()

In [7]:
df_tr_dev_test = pd.read_pickle(f'../data/pkl/{cli_args["data"]}_data.demand_{cli_args["sr_freq"]}_sequences_{len(windowing_params["rnn_feats"])}_rnn_inputs_{len(windowing_params["fc_feats"])}_fc_inputs_{len(windowing_params["y_feats"])}_outputs_length_{windowing_params["length_max"]}_stride_{windowing_params["stride"]}.v4.pickle')

In [8]:
building_tokens, building_token_lookup = (
    pd.read_pickle(f'../data/pkl/{cli_args["data"]}_data_12H_building_tokens_v4.pkl'), 
    pd.read_pickle(f'../data/pkl/{cli_args["data"]}_data_12H_building_token_lookup_v4.pkl')
)
model_tokens, model_token_lookup = (
    pd.read_pickle(f'../data/pkl/{cli_args["data"]}_data_12H_model_tokens_v4.pkl'), 
    pd.read_pickle(f'../data/pkl/{cli_args["data"]}_data_12H_model_token_lookup_v4.pkl')
)

In [9]:
df_evcs_meta = pd.read_pickle(f'../data/pkl/{cli_args["data"]}_data.metadata.v3.pickle')

# Load Model

In [10]:
building_embeddings = torch.nn.Embedding(len(building_token_lookup), 15) 
model_embeddings = torch.nn.Embedding(len(model_token_lookup), 3) 

In [11]:
model_params = dict(
    location_embeddings=building_embeddings,
    model_embeddings=model_embeddings,
    input_size=len(windowing_params['rnn_feats']),
    scale=None,
    rnn_cell=getattr(torch.nn, cli_args['rnn_cell'].upper()),
    bidirectional=cli_args['bi'],
    num_layers=cli_args['num_layers'],
    hidden_size=cli_args['hidden_size'],
    misc_size=len(windowing_params['fc_feats']),
    fc_layers=[int(i) for i in cli_args['fc_layers'].split(',')],
    output_size=len(windowing_params['y_feats']),
)

In [12]:
MODEL_RNN_TYPE_NAME = f'{"Bi" if model_params["bidirectional"] else ""}{cli_args["rnn_cell"].upper()}'

model_name_base = f'{"bi-" if model_params["bidirectional"] else ""}'+\
                    f'{cli_args["rnn_cell"]}_{model_params["num_layers"]}_'+\
                    f'{model_params["hidden_size"]}_fc_{"_".join(map(str, model_params["fc_layers"]))}_'+\
                    f"{len(windowing_params['rnn_feats'])}_rnn_inputs_{len(windowing_params['fc_feats'])}_fc_inputs_{len(windowing_params['y_feats'])}_outputs_"+\
                    f'batchsize_{cli_args["bs"]}_patience_{cli_args["patience"]}__'+\
                    f'{cli_args["data"]}_dataset_{cli_args["sr_freq"]}_sequences_'+\
                    f'window_{windowing_params["length_max"]}_stride_{windowing_params["stride"]}.{"PinballLoss_q70"}.dropout_after_cat.cml.epoch{{0}}.pth'
                    
save_path_best = os.path.join('..', 'data', 'pth', f'{model_name_base.split(".")[0]}', model_name_base.format('best'))
print(save_path_best)

../data/pth/bi-gru_1_24_fc_24_14_rnn_inputs_1_fc_inputs_1_outputs_batchsize_32_patience_10__paloalto_dataset_12H_sequences_window_48_stride_1/bi-gru_1_24_fc_24_14_rnn_inputs_1_fc_inputs_1_outputs_batchsize_32_patience_10__paloalto_dataset_12H_sequences_window_48_stride_1.PinballLoss_q70.dropout_after_cat.cml.epochbest.pth


In [13]:
def load_model(path, model_config, device):
    # # Evaluate Best Model
    checkpoint = torch.load(path, map_location=device)

    fededf_model = ml.EnergyDemandForecasting_v2(
        **model_config
    )
    fededf_model.to(device)

    # Assign each NumPy array to the corresponding layer in the model
    with torch.no_grad():  # Disable gradient tracking to avoid issues during assignment
        for param, np_array in zip(fededf_model.parameters(), checkpoint['model_state_dict'].values()):
            # Convert NumPy array to a torch tensor with the same dtype as model parameters
            param.copy_(torch.tensor(np_array, dtype=param.dtype))

    fededf_model.eval()
    return fededf_model

# Make Predictions

In [14]:
# # Create features' temporal sequence (i.e. training dataset)
identity_function = FunctionTransformer(None) # We do not need a scaler for now, so we use the Identity function...
evcs_dataset_windows_test = df_tr_dev_test.xs(3, level=1, drop_level=False).dropna().copy()

test_dataset = ds.EDFDataset_v2(building_tokens, model_tokens, building_token_lookup, model_token_lookup, evcs_dataset_windows_test, scaler=identity_function)
test_loader = DataLoader(test_dataset,  batch_size=1, shuffle=False, collate_fn=test_dataset.pad_collate)

In [15]:
def fededf_model_inference(model, data_loader):
    y_true_oid_nn, y_pred_oid_nn = [], []

    with torch.no_grad():
        for xb, yb, lb, *args in (pbar := tqdm.tqdm(data_loader, leave=False, total=len(data_loader), dynamic_ncols=True)):
            # print(f'{xb.shape=}\t {yb.shape=}\t {lb.shape=}')
            xb, yb = xb.to(device), yb.to(device)    # Model Inference
            args = (arg.to(device) for arg in args)
            y_pred = model(xb.float(), lb, *args).detach()
            
            y_true_oid_nn.append(yb)
            y_pred_oid_nn.append(y_pred)

    y_true_oid_nn, y_pred_oid_nn = np.concatenate(y_true_oid_nn), np.concatenate(y_pred_oid_nn)
    return y_true_oid_nn, y_pred_oid_nn

In [16]:
def fededf_model_inference_table(evcs_dataset_windows_test, df_evcs_meta, y_true_oid_nn, y_pred_oid_nn, t_horizon=0):
    evcs_dataset_windows_y_pred_time_axis = evcs_dataset_windows_test.time_axes.apply(lambda l: l[-1]).values

    lstm_results = pd.concat(
        {
            MODEL_RNN_TYPE_NAME:pd.Series(
                y_pred_oid_nn[:, t_horizon, :].squeeze(),
                index=[
                    evcs_dataset_windows_test.index.get_level_values(0),
                    evcs_dataset_windows_y_pred_time_axis
                ],
            ).rename_axis(['oid', 'timestamp']),
            'power_next':pd.Series(
                y_true_oid_nn[:, t_horizon, :].squeeze(),
                index=[
                    evcs_dataset_windows_test.index.get_level_values(0),
                    evcs_dataset_windows_y_pred_time_axis
                ]
            ).rename_axis(['oid', 'timestamp']),
        }, 
        axis=1
    )

    lstm_results = lstm_results.loc[~lstm_results.index.duplicated(keep='last')].copy() # Drop duplicated entries (in case of overlapping windows)
    lstm_results = lstm_results.unstack('timestamp')

    return lstm_results.groupby('oid', group_keys=False).apply(
        lambda l: l * (df_evcs_meta.loc[l.name, 'power_output_kW'] * (pd.Timedelta(SR_FREQ).total_seconds() / 3600))
    )

In [17]:
fededf_global_epoch_i = load_model(
    save_path_best,
    model_params,
    device
)

y_true_oid_nn, y_pred_oid_nn = fededf_model_inference(fededf_global_epoch_i, test_loader)

edf_results = fededf_model_inference_table(evcs_dataset_windows_test, df_evcs_meta, y_true_oid_nn, y_pred_oid_nn, t_horizon=0)

/Users/andrewt/miniforge3/envs/torch/lib/python3.10/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator FunctionTransformer from version 1.4.2 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/andrewt/Documents/DataStories-UniPi/FedEDF/src/models.py:44: UserWarning: Instantiated instance without standardization. Falling back to identity function...
  warnings.warn("Instantiated instance without standardization. Falling back to identity function...")
/var/folders/p6/kw8t6dgj12v1289kljz8mz4w0000gn/T/ipykernel_50905/3623601349.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  param.copy_(torch.tensor(np

In [18]:
available_oids = list(
    set(
        df_tr_dev_test.xs(3, level=1).dropna().index.unique()

    ).intersection(
        set(
            df_tr_dev_test.xs(1, level=1).dropna().index.unique()
        )
    )
)

train_set_max_timestamp = str(df_tr_dev_test.loc[pd.IndexSlice[:, 1], 'time_axes'].explode().max().date())

y_train = df_evse_demand_v3.loc[pd.IndexSlice[:, available_oids, :train_set_max_timestamp], 'power_curr']
oid_indices = df_evse_demand_v3.loc[pd.IndexSlice[:, available_oids, :train_set_max_timestamp]].sort_index().groupby('oid', observed=False).groups

In [19]:
edf_results_final = edf_results.clip(lower=0).stack(level=1, future_stack=True)

model_results_metrics = hl.evaluate_predictions(
    edf_results_final.loc[
        pd.IndexSlice[available_oids, :], :
    ].dropna(),
    y_true_name='power_next',
    y_pred_names=[MODEL_RNN_TYPE_NAME],
    eval_funs=[
        ('MASE_pct', MeanAbsoluteScaledError(sp=24), {'y_train':y_train, 'oid_indices':oid_indices}),
        ('SMAPE_pct', MeanAbsolutePercentageError(symmetric=True), {}),
        ('MAAPE_rads', hl.mean_arctangent_absolute_percentage_error, {}),
        ('WAPE_pct', hl.wape, {}),
        ('RMSE_kW', MeanSquaredError(square_root=True), {}),
        ('MAE_kW', MeanAbsoluteError(), {}),
        ('R2', r2_score, {})
    ]
)

In [20]:
model_results_metrics.groupby(level=0, sort=False).describe().T.loc[
    pd.IndexSlice[:, ['mean', '25%', '50%', '75%']], :
].round(2)

0_MASE_pct  1_SMAPE_pct  2_MAAPE_rads  3_WAPE_pct  4_RMSE_kW  \
BiGRU mean        0.59         1.30          1.05        1.84       9.37   
      25%         0.37         1.11          0.92        0.94       7.76   
      50%         0.61         1.33          1.05        1.12       9.66   
      75%         0.76         1.38          1.12        1.30      11.80   

            5_MAE_kW  6_R2  
BiGRU mean      7.17 -0.07  
      25%       5.57 -0.09  
      50%       7.37  0.17  
      75%       9.33  0.29

In [22]:
edf_results_final.dropna().to_pickle(f'../data/pkl/{cli_args["data"]}_data.demand_{cli_args["sr_freq"]}.EVSE__{MODEL_RNN_TYPE_NAME}__model.v4.pickle')
model_results_metrics.to_pickle(f'../data/pkl/{cli_args["data"]}_data.demand_{cli_args["sr_freq"]}.EVSE__{MODEL_RNN_TYPE_NAME}__model.metrics.v4.pickle')